# ДЗ 2.2. Оптимизация вычислений в Spark

Docker поднимает:
- [S3](http://localhost:9001) (RustFS),
- [Spark](http://localhost:4040) (master + 1 worker: 4 ядра, 4 ГБ)
- [Jupyter](http://localhost:8888/?token=dwh).

Какие данные есть:

- События из ДЗ2.1 в S3
- SCD2 из ДЗ2.1
- Справочник 25 кампаний (`task1/campaigns.csv` от преподавателя)

---
## Задание 1. Сравнение стратегий JOIN

**Входные данные:** SCD2 пользователей (повторный прогон моего `build_scd2.py` из ДЗ 2.1 на 5 млн событий) и справочник 25 кампаний. SCD2 строится и сохраняется в S3 до любых измерений.

In [1]:
!python3 build_scd2.py   # s3a://hw2-2/events/ -> SCD2 в s3a://hw2-2/users_scd2/

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 16:58:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/13 16:58:07 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
Готово: 1080897 строк в s3a://hw2-2/users_scd2/                                 


In [2]:
import os, re, json, time, statistics, urllib.request
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName("hw2_2")
         .master(os.environ["SPARK_MASTER"])
         .config("spark.driver.host", os.environ["SPARK_DRIVER_HOST"])
         .config("spark.driver.bindAddress", "0.0.0.0")
         .config("spark.driver.memory", "1g")
         .config("spark.executor.memory", "3g")
         .config("spark.executor.cores", "4")
         .config("spark.cores.max", "4")              # ровно один executor на 4 ядра
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])
         .config("spark.hadoop.fs.s3a.access.key", os.environ["S3_ACCESS_KEY"])
         .config("spark.hadoop.fs.s3a.secret.key", os.environ["S3_SECRET_KEY"])
         .config("spark.hadoop.fs.s3a.path.style.access", "true")
         .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
         .getOrCreate())
sc, B = spark.sparkContext, "s3a://hw2-2"

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/13 16:58:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
TS = "yyyy-MM-dd'T'HH:mm:ssXXX"
scd2 = spark.read.parquet(f"{B}/users_scd2/")
campaigns = (spark.read.schema("campaign_id INT, email_domain STRING, send_at TIMESTAMP, campaign_name STRING")
             .option("header", True).option("timestampFormat", TS).csv(f"{B}/task1/campaigns.csv"))

print("SCD2:", 
      scd2.count(), "версий,", 
      scd2.select("user_id").distinct().count(), "пользователей,",
      scd2.filter("is_current").count(), "текущих версий")

26/11/13 16:58:41 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


SCD2: 1080897 версий, 100000 пользователей, 100000 текущих версий


### 1. INNER JOIN витрины со справочником

In [4]:
users = scd2.withColumn("user_domain", F.substring_index("user_email", "@", -1))   # домен email версии профиля

def audience(c):
    cond = ((users.user_domain == c.email_domain) & (users.valid_from <= c.send_at)
            & (users.valid_to.isNull() | (c.send_at < users.valid_to)))
    return users.join(c, cond).select("campaign_id", "campaign_name", "send_at", "email_domain",
                                      "user_id", "user_fio", "user_email", "user_login")

audience(campaigns).show(5, truncate=False)

+-----------+-------------+-------------------+---------------+-------+-------------+----------------------------------+----------+
|campaign_id|campaign_name|send_at            |email_domain   |user_id|user_fio     |user_email                        |user_login|
+-----------+-------------+-------------------+---------------+-------+-------------+----------------------------------+----------+
|18         |Campaign_0018|2026-01-11 06:56:44|post.example   |1      |Brittany Reed|john.scott.1@post.example         |jscott1r1 |
|5          |Campaign_0005|2026-01-07 02:27:37|post.example   |1      |Brittany Reed|john.scott.1@post.example         |jscott1r1 |
|19         |Campaign_0019|2026-01-25 05:32:10|webmail.example|1      |Brittany Reed|brittany.reed.1.r3@webmail.example|jscott1r1 |
|4          |Campaign_0004|2026-02-17 23:41:49|webmail.example|1      |Olivia Murphy|brittany.reed.1.r3@webmail.example|omurphy1r7|
|3          |Campaign_0003|2026-02-15 01:56:37|webmail.example|1      |Olivi

### 2. Четыре варианта одного запроса
Бизнес-логика одна (`audience`), меняется только способ JOIN.

In [5]:
VARIANTS = {"auto":         lambda: audience(campaigns),
            "merge":        lambda: audience(campaigns.hint("merge")),
            "broadcast":    lambda: audience(F.broadcast(campaigns)),
            "shuffle_hash": lambda: audience(campaigns.hint("shuffle_hash"))}

In [6]:
FIXED = {"spark.sql.adaptive.enabled": "false", 
         "spark.sql.autoBroadcastJoinThreshold": "-1",
         "spark.sql.shuffle.partitions": "16"}

DEFAULT = {k: spark.conf.get(k) for k in FIXED}       # исходные настройки – для auto
def configure(auto):
    for k, v in (DEFAULT if auto else FIXED).items():
        spark.conf.set(k, v)

RES1 = f"{B}/results/task1"
print("исходные настройки:", DEFAULT)

исходные настройки: {'spark.sql.adaptive.enabled': 'true', 'spark.sql.autoBroadcastJoinThreshold': '10485760b', 'spark.sql.shuffle.partitions': '200'}


### 3. Фактическая стратегия по физическому плану и сравнение результатов
Каждый вариант записываем в S3 и берём **конечный** физический план выполненного запроса из Spark UI.

In [7]:
UI = f"http://localhost:4040/api/v1/applications/{sc.applicationId}"   # REST API Spark UI
api = lambda path: json.load(urllib.request.urlopen(f"{UI}/{path}"))
JOINS = r"(BroadcastHashJoin|SortMergeJoin|ShuffledHashJoin)"

def last_sql():
    # последний выполненный запрос во вкладке SQL: план и id его jobs
    time.sleep(1)                                     # UI дописывает данные асинхронно
    last = max(e["id"] for e in api("sql?details=false&planDescription=false&length=100000"))
    return api(f"sql/{last}")

for name, make in VARIANTS.items():
    configure(auto=(name == "auto"))
    make().write.mode("overwrite").parquet(f"{RES1}/{name}")
    plan = last_sql()["planDescription"]
    print(f"===== {name}: {sorted(set(re.findall(JOINS, plan)))}\n{plan.split(chr(10) * 3)[0]}\n")
configure(auto=True)

===== auto: ['BroadcastHashJoin']
== Physical Plan ==
AdaptiveSparkPlan (22)
+- == Final Plan ==
   Execute InsertIntoHadoopFsRelationCommand (12)
   +- WriteFiles (11)
      +- * Project (10)
         +- * BroadcastHashJoin Inner BuildRight (9)
            :- * Project (4)
            :  +- * Filter (3)
            :     +- * ColumnarToRow (2)
            :        +- Scan parquet  (1)
            +- BroadcastQueryStage (8), Statistics(sizeInBytes=1025.0 KiB, rowCount=25)
               +- BroadcastExchange (7)
                  +- * Filter (6)
                     +- Scan csv  (5)
+- == Initial Plan ==
   Execute InsertIntoHadoopFsRelationCommand (21)
   +- WriteFiles (20)
      +- Project (19)
         +- BroadcastHashJoin Inner BuildRight (18)
            :- Project (15)
            :  +- Filter (14)
            :     +- Scan parquet  (13)
            +- BroadcastExchange (17)
               +- Filter (16)
                  +- Scan csv  (5)



===== merge: ['SortMergeJoin']
== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (14)
+- WriteFiles (13)
   +- * Project (12)
      +- * SortMergeJoin Inner (11)
         :- * Sort (6)
         :  +- Exchange (5)
         :     +- * Project (4)
         :        +- * Filter (3)
         :           +- * ColumnarToRow (2)
         :              +- Scan parquet  (1)
         +- * Sort (10)
            +- Exchange (9)
               +- * Filter (8)
                  +- Scan csv  (7)



===== broadcast: ['BroadcastHashJoin']
== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (11)
+- WriteFiles (10)
   +- * Project (9)
      +- * BroadcastHashJoin Inner BuildRight (8)
         :- * Project (4)
         :  +- * Filter (3)
         :     +- * ColumnarToRow (2)
         :        +- Scan parquet  (1)
         +- BroadcastExchange (7)
            +- * Filter (6)
               +- Scan csv  (5)



===== shuffle_hash: ['ShuffledHashJoin']
== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (12)
+- WriteFiles (11)
   +- * Project (10)
      +- * ShuffledHashJoin Inner BuildRight (9)
         :- Exchange (5)
         :  +- * Project (4)
         :     +- * Filter (3)
         :        +- * ColumnarToRow (2)
         :           +- Scan parquet  (1)
         +- Exchange (8)
            +- * Filter (7)
               +- Scan csv  (6)



Сравнение с `auto` полностью, с учётом дубликатов: `exceptAll` в обе стороны. Для пары `(campaign_id, user_id)` не больше одной строки.

In [8]:
out1 = {n: spark.read.parquet(f"{RES1}/{n}") for n in VARIANTS}
ref = out1["auto"]
for n, df in out1.items():
    dup = df.groupBy("campaign_id", "user_id").count().filter("count > 1").count()
    diff = df.exceptAll(ref).count() + ref.exceptAll(df).count()
    print(f"{n:13} строк={df.count():>8}  файлов={len(df.inputFiles()):>3}  повторов (campaign_id,user_id)={dup}  отличий от auto={diff}")

auto          строк=  500233  файлов=  2  повторов (campaign_id,user_id)=0  отличий от auto=0


merge         строк=  500233  файлов=  5  повторов (campaign_id,user_id)=0  отличий от auto=0


broadcast     строк=  500233  файлов=  2  повторов (campaign_id,user_id)=0  отличий от auto=0


shuffle_hash  строк=  500233  файлов=  5  повторов (campaign_id,user_id)=0  отличий от auto=0


Все четыре варианта вернули по 500233 строк. `exceptAll` в обе стороны относительно `auto` даёт 0 отличий, то есть результаты совпадают полностью, с учётом повторов. Повторяющихся пар `(campaign_id, user_id)` нет ни в одном варианте.

### 4. Измерения
Окружение и ресурсы одинаковые для всех вариантов.

In [9]:
conf = sc.getConf()
cpu = next(l.split(":")[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))   # CPU хоста
ram = round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1)                  # память Docker VM, ГБ
ENV = {"spark": spark.version,
       "executor": f'1 x {conf.get("spark.executor.cores")} ядра, {conf.get("spark.executor.memory")}',
       "driver": conf.get("spark.driver.memory"),
       "host": f"{cpu}, {os.cpu_count()} vCPU, {ram} ГБ"}
print(ENV)

{'spark': '3.5.3', 'executor': '1 x 4 ядра, 3g', 'driver': '1g', 'host': '12th Gen Intel(R) Core(TM) i7-12700H, 20 vCPU, 7.6 ГБ'}


**Функция замера**

In [10]:
MB = lambda b: round(b / 2**20, 1)   # байты -> МБ

def measure(df, path, variant):
    t0 = time.perf_counter()
    df.write.mode("overwrite").parquet(path)              # полная запись результата
    sec = round(time.perf_counter() - t0, 2)

    q = last_sql()                                        # запрос во вкладке SQL: конечный план и его jobs
    stages = [a for j in q["successJobIds"]               # все стадии всех jobs запроса
              for s in api(f"jobs/{j}")["stageIds"] for a in api(f"stages/{s}")]
    heavy = max(stages, key=lambda a: (a["shuffleReadBytes"], a["executorRunTime"]))   # стадия JOIN (читает shuffle); без shuffle – самая долгая
    t = api(f"stages/{heavy['stageId']}/{heavy['attemptId']}/taskSummary?quantiles=0.5,1.0")["executorRunTime"]

    return dict(variant=variant, sec=sec,
                join=",".join(sorted(set(re.findall(JOINS, q["planDescription"])))),   # фактический оператор JOIN
                aqe=spark.conf.get("spark.sql.adaptive.enabled"),                     # настройки, с которыми шёл запуск
                partitions=spark.conf.get("spark.sql.shuffle.partitions"),
                shuffle_read_MB=MB(sum(a["shuffleReadBytes"] for a in stages)),
                shuffle_write_MB=MB(sum(a["shuffleWriteBytes"] for a in stages)),
                mem_spill_MB=MB(sum(a["memoryBytesSpilled"] for a in stages)),        # Memory Bytes Spilled
                disk_spill_MB=MB(sum(a["diskBytesSpilled"] for a in stages)),         # Disk Bytes Spilled
                heavy_stage_tasks=heavy["numTasks"],                                  # = число партиций стадии
                task_median_s=round(t[0] / 1000, 1),                                  # медиана времени задачи
                task_max_s=round(t[1] / 1000, 1),                                     # самая долгая задача
                **ENV), q["planDescription"]                                          # окружение – в каждую строку

**Сводная таблица**: по каждому варианту все замеры, медиана, минимум и максимум.

In [11]:
def summary(runs):
    rows = []
    for v in dict.fromkeys(r["variant"] for r in runs):          # варианты в порядке появления
        rs = [r for r in runs if r["variant"] == v]
        s = [r["sec"] for r in rs]
        rows.append({**rs[-1],                                   # стратегия и метрики последнего запуска
                     "sec": round(statistics.median(s), 2), "min": min(s), "max": max(s),
                     "all_runs": " ".join(map(str, s))})
    return spark.createDataFrame([tuple(r.values()) for r in rows], list(rows[0]))

Прогрев уже был в п. 3, здесь 5 замеров каждого варианта с чередованием порядка.

In [12]:
runs1, names = [], list(VARIANTS)
for i in range(5):
    for name in names[i % 4:] + names[:i % 4]:        # раунд i начинается с i-го варианта: auto,merge,… → merge,…,auto → …
        configure(auto=(name == "auto"))              # auto с исходными настройками, остальные по условию: AQE off, без autoBroadcast, 16 партиций
        runs1.append(measure(VARIANTS[name](), 
                             f"{RES1}/{name}", name)[0])
configure(auto=True)
summary(runs1).show(truncate=False)

+------------+----+-----------------+-----+----------+---------------+----------------+------------+-------------+-----------------+-------------+----------+-----+--------------+------+-----------------------------------------------------+----+----+------------------------+
|variant     |sec |join             |aqe  |partitions|shuffle_read_MB|shuffle_write_MB|mem_spill_MB|disk_spill_MB|heavy_stage_tasks|task_median_s|task_max_s|spark|executor      |driver|host                                                 |min |max |all_runs                |
+------------+----+-----------------+-----+----------+---------------+----------------+------------+-------------+-----------------+-------------+----------+-----+--------------+------+-----------------------------------------------------+----+----+------------------------+
|auto        |1.73|BroadcastHashJoin|true |200       |0.0            |0.0             |0.0         |0.0          |4                |0.1          |1.1       |3.5.3|1 x 4 ядра, 

> Какой вариант оказался быстрее и насколько результат устойчив относительно разброса?

Быстрее варианты без shuffle: auto (медиана 1,73 с) и broadcast (1,82 с) - примерно на 0,6-0,8 с (~30%) быстрее shuffle_hash (2,44 с) и merge (2,50 с). По медианам разница устойчивая, но разброс большой: в третьем раунде все варианты были медленнее (от 2,45 до 3,78 с) - это общий шум стенда, а разница между auto и broadcast (0,09 с) в пределах этого шума.

> Почему меньший shuffle мог не дать меньшего времени чтения, JOIN и записи?

Shuffle здесь всего 48 МБ и без spill, поэтому выигрыш от его отсутствия в абсолютных цифрах меньше секунды: основное время занимают чтение SCD2 из S3, запись 500 тыс. строк в Parquet на S3 с коммитом файлов и фиксированные накладные расходы на запуск jobs, а эти этапы у всех вариантов одинаковые.

> Какой план выбрал Spark без hints? Изменил ли явный broadcast этот план?

Без hints Spark выбрал BroadcastHashJoin, так как справочник из 25 строк (в broadcast ~1 МБ, см. `Statistics` в плане) меньше порога autoBroadcast 10 МБ; явный broadcast дал тот же оператор BroadcastHashJoin Inner BuildRight и отличается только отсутствием обёртки AQE (AdaptiveSparkPlan), потому что AQE в этом варианте выключен.

> Как пять доменов влияют на распределение данных? При каких изменениях нагрузки результат может стать другим?

Ключ JOIN - домен, а различных доменов всего 5, поэтому при shuffle все строки одного домена попадают в одну партицию: заняты максимум 5 из 16 партиций, и сколько бы партиций и ядер ни было, параллельно работают не больше 5 задач, а broadcast от этого не зависит, потому что SCD2 не перемешивается.

Результат может измениться, если:
- SCD2 сильно вырастет (5 огромных партиций начнут спиллить и broadcast выиграет сильнее),
- если справочник перестанет помещаться в память (broadcast станет невозможен),
- если один домен станет доминировать (перекос)
- если доменов станет много (shuffle распределится равномерно и разница сократится).

Выбор стратегии для стенда:
Broadcast справочника кампаний: он крошечный, shuffle не нужен, нет зависимости от числа доменов, и именно его Spark выбирает сам.

---
## Задание 2. Точно ли всё так хорошо?

- Просмотры: `skew_transactions.csv` (`user_uid, element_uid, watched_time`).
- Каталог: `catalogue.json`, берём `type` и `duration`.
- Ключ соединения - `element_uid`.

In [13]:
tx = spark.read.csv(f"{B}/task2/skew_transactions.csv", header=True, schema="user_uid INT, element_uid INT, watched_time INT")
cat = (spark.read.text(f"{B}/task2/catalogue.json", wholetext=True)
       .select(F.explode(F.from_json("value", "map<string, struct<type:string, duration:int>>")).alias("element_uid", "m"))
       .select(F.col("element_uid").cast("int").alias("element_uid"), "m.type", "m.duration"))

RES2 = f"{B}/results/task2"

### 1. Исходный INNER JOIN
Исходный вариант запускаем как обычный shuffle-join, с теми же настройками, что в задании 1 (AQE off, autoBroadcast -1, 16 партиций). Так JOIN ведёт себя, когда справочник не помещается в broadcast.

In [14]:
def enrich(t, c, keys=["element_uid"]):
    return t.join(c, keys).select("user_uid", "element_uid", "watched_time", "type", "duration")

configure(auto=True)
enrich(tx, cat).explain()                       # план с исходными настройками

configure(auto=False)
r_orig, plan_orig = measure(enrich(tx, cat), f"{RES2}/original", "original")
print(plan_orig.split(chr(10) * 3)[0])

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [user_uid#1780, element_uid#1781, watched_time#1782, type#1794, duration#1795]
   +- BroadcastHashJoin [element_uid#1781], [element_uid#1793], Inner, BuildRight, false
      :- Filter isnotnull(element_uid#1781)
      :  +- FileScan csv [user_uid#1780,element_uid#1781,watched_time#1782] Batched: false, DataFilters: [isnotnull(element_uid#1781)], Format: CSV, Location: InMemoryFileIndex(1 paths)[s3a://hw2-2/task2/skew_transactions.csv], PartitionFilters: [], PushedFilters: [IsNotNull(element_uid)], ReadSchema: struct<user_uid:int,element_uid:int,watched_time:int>
      +- BroadcastExchange HashedRelationBroadcastMode(List(cast(input[0, int, true] as bigint)),false), [plan_id=4434]
         +- Project [cast(element_uid#1788 as int) AS element_uid#1793, m#1789.type AS type#1794, m#1789.duration AS duration#1795]
            +- Filter isnotnull(cast(element_uid#1788 as int))
               +- Generate explode(from_json(MapT

== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (14)
+- WriteFiles (13)
   +- * Project (12)
      +- * SortMergeJoin Inner (11)
         :- * Sort (4)
         :  +- Exchange (3)
         :     +- * Filter (2)
         :        +- Scan csv  (1)
         +- * Sort (10)
            +- Exchange (9)
               +- * Project (8)
                  +- * Filter (7)
                     +- Generate (6)
                        +- Scan text  (5)


### 2. Диагностика
Частоты ключей и доля самых частых, NULL, уникальность ключа каталога, распределение строк по shuffle-партициям и время задач Spark.

In [15]:
total = tx.count()
freq = tx.groupBy("element_uid").count()
freq.orderBy(F.desc("count")).limit(5).withColumn("share_%", F.round(100 * F.col("count") / total, 2)).show()
print(f"просмотров: {total}, различных element_uid: {freq.count()}, NULL element_uid: {tx.filter('element_uid IS NULL').count()}")

+-----------+--------+-------+
|element_uid|   count|share_%|
+-----------+--------+-------+
|       2714|20496000|  68.16|
|        747|   61272|    0.2|
|       2639|   61119|    0.2|
|       3783|   55228|   0.18|
|       2245|   52942|   0.18|
+-----------+--------+-------+



просмотров: 30070692, различных element_uid: 8296, NULL element_uid: 0


In [16]:
print(f"каталог: {cat.count()} строк, {cat.select('element_uid').distinct().count()} уникальных ключей, "
      f"NULL ключей: {cat.filter('element_uid IS NULL').count()}")

каталог: 10200 строк, 10200 уникальных ключей, NULL ключей: 0


In [17]:
# Признаки перекоса: куда попадёт каждая строка при hash-партиционировании на 16 партиций
tx.groupBy(F.pmod(F.hash("element_uid"), F.lit(16)).alias("partition")).count().orderBy(F.desc("count")).show(5)

print("стадия JOIN исходного варианта:", {k: r_orig[k] for k in ("heavy_stage_tasks", "task_median_s", "task_max_s", "mem_spill_MB", "disk_spill_MB")})

+---------+--------+
|partition|   count|
+---------+--------+
|        8|21205547|
|        0|  728933|
|       14|  700290|
|        5|  687953|
|       13|  684970|
+---------+--------+
only showing top 5 rows

стадия JOIN исходного варианта: {'heavy_stage_tasks': 16, 'task_median_s': 0.7, 'task_max_s': 12.8, 'mem_spill_MB': 240.0, 'disk_spill_MB': 78.1}


Ключ `element_uid = 2714` даёт 68% всех просмотров (20,5 млн из 30 млн), остальные ключи - до 0,2%. 

NULL нет, ключ каталога уникален, так что проблема не в данных, а в перекосе: при shuffle все строки 2714 уходят в одну партицию (21,2 млн строк против ~0,6 млн в остальных). Поэтому одна задача JOIN идёт ~13 с при медиане 0,7 с и уходит в spill, а остальные ядра в это время простаивают.

### 3. Решение: broadcast каталога
Каталог маленький: около 10 тыс. строк и ~1 МБ в памяти (размер broadcast в Spark UI) против 3 ГБ у executor, ключ уникален. Его можно целиком разослать на executor: просмотры не перемешиваются, и горячий ключ ни на что не влияет. Настройки те же, что у исходного варианта; broadcast задан явно.

In [18]:
r_b, plan_b = measure(enrich(tx, F.broadcast(cat)), f"{RES2}/broadcast", "broadcast")
print(plan_b.split(chr(10) * 3)[0])
summary([r_orig, r_b]).show(truncate=False)

== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (11)
+- WriteFiles (10)
   +- * Project (9)
      +- * BroadcastHashJoin Inner BuildRight (8)
         :- * Filter (2)
         :  +- Scan csv  (1)
         +- BroadcastExchange (7)
            +- * Project (6)
               +- * Filter (5)
                  +- Generate (4)
                     +- Scan text  (3)
+---------+-----+-----------------+-----+----------+---------------+----------------+------------+-------------+-----------------+-------------+----------+-----+--------------+------+-----------------------------------------------------+-----+-----+--------+
|variant  |sec  |join             |aqe  |partitions|shuffle_read_MB|shuffle_write_MB|mem_spill_MB|disk_spill_MB|heavy_stage_tasks|task_median_s|task_max_s|spark|executor      |driver|host                                                 |min  |max  |all_runs|
+---------+-----+-----------------+-----+----------+---------------+----------------+------------+-------

Broadcast убрал shuffle целиком (0 МБ вместо 359 МБ) и spill, перекоса больше нет, и запись стала почти в 2 раза быстрее: ~13 с против ~25 с. Цена - каталог копируется на каждый executor, но это всего ~10 тыс. строк.

### 4. Salt Join
Соль добавляем только горячим ключам - тем, у кого больше 1% всех просмотров (это один `element_uid = 2714`, у остальных меньше 0,2%). 

Значений соли `N = 16` = число shuffle-партиций: горячий ключ делится на \~16 кусков по \~1,3 млн строк, это сопоставимо с обычной партицией (~0,6 млн).
* в просмотрах горячий ключ получает соль `pmod(hash(user_uid, watched_time), N)` - детерминированную, поэтому при перезапуске задачи строки попадут туда же; обычные ключи - соль 0;
* в каталоге строка горячего ключа размножается в N копий с солью `0..N-1`, у обычных ключей одна копия с солью 0.

Каждая строка просмотров находит ровно одну копию своей строки каталога - все исходные совпадения сохраняются, лишних нет.

In [19]:
HOT_SHARE, N = 0.01, 16
hot = [r.element_uid for r in freq.filter(F.col("count") > HOT_SHARE * total).collect()]
is_hot = F.col("element_uid").isin(hot)
tx_s = tx.withColumn("salt", F.when(is_hot, F.pmod(F.hash("user_uid", "watched_time"), F.lit(N))).otherwise(0))
cat_s = cat.withColumn("salt", F.explode(F.when(is_hot, F.sequence(F.lit(0), F.lit(N - 1))).otherwise(F.array(F.lit(0)))))
print("горячие ключи:", hot, f"| каталог: {cat.count()} -> {cat_s.count()} строк после соли")

горячие ключи: [2714] | каталог: 10200 -> 10215 строк после соли


In [20]:
tx_s.filter(is_hot).groupBy("salt").count().orderBy("salt").show(N)

+----+-------+
|salt|  count|
+----+-------+
|   0|1279200|
|   1|1273800|
|   2|1291500|
|   3|1285500|
|   4|1272300|
|   5|1302300|
|   6|1286400|
|   7|1291500|
|   8|1270500|
|   9|1273500|
|  10|1312800|
|  11|1266900|
|  12|1291500|
|  13|1258500|
|  14|1247100|
|  15|1292700|
+----+-------+



In [21]:
r_s, plan_s = measure(enrich(tx_s, cat_s, ["element_uid", "salt"]), f"{RES2}/salt", "salt")
print(plan_s.split(chr(10) * 3)[0])

== Physical Plan ==
Execute InsertIntoHadoopFsRelationCommand (16)
+- WriteFiles (15)
   +- * Project (14)
      +- * SortMergeJoin Inner (13)
         :- * Sort (5)
         :  +- Exchange (4)
         :     +- * Project (3)
         :        +- * Filter (2)
         :           +- Scan csv  (1)
         +- * Sort (12)
            +- Exchange (11)
               +- * Generate (10)
                  +- * Project (9)
                     +- * Filter (8)
                        +- Generate (7)
                           +- Scan text  (6)


### 5. Сравнение трёх вариантов
Настройки одинаковые (AQE off, autoBroadcast -1, 16 партиций), по 3 замера с чередованием порядка. `task_median_s` / `task_max_s` - по самой тяжёлой стадии: при перекосе максимум сильно больше медианы.

In [22]:
VARIANTS2 = {"original":  lambda: enrich(tx, cat),
             "broadcast": lambda: enrich(tx, F.broadcast(cat)),
             "salt":      lambda: enrich(tx_s, cat_s, ["element_uid", "salt"])}
configure(auto=False)                          # AQE off, без autoBroadcast, 16 партиций - как в п. 1-4
runs2, names2 = [], list(VARIANTS2)
for i in range(3):
    for name in names2[i % 3:] + names2[:i % 3]:
        runs2.append(measure(VARIANTS2[name](), f"{RES2}/{name}", name)[0])
configure(auto=True)
summary(runs2).show(truncate=False)

+---------+-----+-----------------+-----+----------+---------------+----------------+------------+-------------+-----------------+-------------+----------+-----+--------------+------+-----------------------------------------------------+-----+-----+-----------------+
|variant  |sec  |join             |aqe  |partitions|shuffle_read_MB|shuffle_write_MB|mem_spill_MB|disk_spill_MB|heavy_stage_tasks|task_median_s|task_max_s|spark|executor      |driver|host                                                 |min  |max  |all_runs         |
+---------+-----+-----------------+-----+----------+---------------+----------------+------------+-------------+-----------------+-------------+----------+-----+--------------+------+-----------------------------------------------------+-----+-----+-----------------+
|original |24.92|SortMergeJoin    |false|16        |359.3          |359.3           |240.0       |78.1         |16               |0.6          |12.1      |3.5.3|1 x 4 ядра, 3g|1g    |12th Gen Inte

Полное совпадение результатов с учётом повторов (`exceptAll` в обе стороны):

In [23]:
out2 = {n: spark.read.parquet(f"{RES2}/{n}") for n in VARIANTS2}
for n in ("broadcast", "salt"):
    a, b = out2["original"], out2[n]
    print(f"{n:10} строк={b.count()} (original={a.count()})  "
          f"original-{n}={a.exceptAll(b).count()}  {n}-original={b.exceptAll(a).count()}")

broadcast  строк=30070692 (original=30070692)  original-broadcast=0  broadcast-original=0


salt       строк=30070692 (original=30070692)  original-salt=0  salt-original=0


### Выводы по заданию 2
Исходный JOIN тормозит из-за одного ключа 2714: все его строки попадают в одну задачу, она работает \~12 с при медиане 0,6 с и спиллит, пока остальные ждут. Лучше всего помог broadcast (медиана \~13,5 с против \~25 с), потому что каталог маленький и shuffle не нужен.

Salt Join тоже убрал перекос (самая долгая задача \~3,4 с, без spill), но вышел медленнее broadcast (\~19 с): 
- shuffle остаётся и даже чуть растёт из-за колонки salt (384 МБ против 359),
- строка горячего ключа в каталоге копируется 16 раз (10 200 -> 10 215 строк).

Поэтому salt нужен, когда справочник не влезает в broadcast.

Результаты broadcast и salt полностью совпадают с исходным с учётом повторов: по 30 070 692 строки, `exceptAll` в обе стороны даёт 0.

---
## Задание 3. Пользовательские признаки

### 1-3. Признаки для выборки
Для каждого пользователя из `ids.csv` min и max `watched_time` (в исходных единицах, секундах) по контенту с его оценкой 8, 9 и 10. 

Колонки `feat_{функция}_watched_time_rating_{оценка}`, связь оценок и просмотров по `(user_uid, element_uid)`. 

Все пользователи выборки сохраняются (LEFT JOIN от выборки); нет подходящих просмотров для оценки - оба её признака NULL, не 0.

In [24]:
ratings = spark.read.csv(f"{B}/task3/cut_ratings.csv", header=True, schema="user_uid INT, element_uid INT, rating INT, ts DOUBLE")
views = spark.read.csv(f"{B}/task3/cut_transactions.csv", header=True, schema="element_uid INT, user_uid INT, watched_time INT")
ids = spark.read.csv(f"{B}/task3/ids.csv", header=True, schema="user_uid INT")

In [25]:
sample = ids.distinct()
r = (ratings.join(F.broadcast(sample), "user_uid", "left_semi")
     .groupBy("user_uid", "element_uid").agg(F.max_by("rating", "ts").alias("rating"))
     .filter(F.col("rating").isin(8, 9, 10)))
v = (views.join(F.broadcast(sample), "user_uid", "left_semi")
     .groupBy("user_uid", "element_uid").agg(F.sum("watched_time").alias("watched_time")))

In [26]:
aggs = [F.expr(f"{f}(CASE WHEN rating = {k} THEN watched_time END)").alias(f"feat_{f}_watched_time_rating_{k}")
        for k in (8, 9, 10) for f in ("min", "max")]

features = sample.join(r.join(v, ["user_uid", "element_uid"]).groupBy("user_uid").agg(*aggs), "user_uid", "left")
features.write.mode("overwrite").parquet(f"{B}/results/task3_features/")

In [27]:
features = spark.read.parquet(f"{B}/results/task3_features/")
features.orderBy("user_uid").show(10)

+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|    8504|                          NULL|                          NULL|                          NULL|                          NULL|                           NULL|                           NULL|
|   24124|                            16|                            16|                          1575|                          2578|                           NULL|                           NULL|
|   2

### 4. Ключи и повторы во входах, принятые решения

In [28]:
for name, df, key in [("ratings", ratings, ["user_uid", "element_uid"]), ("views", views, ["user_uid", "element_uid"]),
                      ("ids", ids, ["user_uid"])]:
    nulls = df.filter(" OR ".join(f"{c} IS NULL" for c in df.columns)).count()
    print(f"{name:8} строк={df.count():>7}  уникальных ключей {key}={df.select(*key).distinct().count():>7}  строк с NULL={nulls}")

ratings  строк= 438790  уникальных ключей ['user_uid', 'element_uid']= 438790  строк с NULL=0
views    строк= 274385  уникальных ключей ['user_uid', 'element_uid']= 274385  строк с NULL=0
ids      строк=    100  уникальных ключей ['user_uid']=    100  строк с NULL=0


Повторов по ключам нет, код защищён от них: 

- если пользователь оценил один фильм несколько раз, берётся последняя оценка (`max_by(rating, ts)`), а несколько просмотров одного фильма суммируются в общее время.
- обе таблицы приводятся к одной строке на `(user_uid, element_uid)`, поэтому JOIN не размножает строки.

**Порядок:** 
1. Сужаем данные до выборки (semi join со 100 id, `ids` маленькая и уходит в broadcast) и до оценок 8-10,
2. Соединяем оценки с просмотрами,
3. Считаем условные агрегаты (`min/max` от `CASE WHEN rating = r`) за один `groupBy`
4. Делаем LEFT JOIN от выборки, чтобы сохранить всех пользователей: у кого нет данных, получат NULL, а не 0.

### Проверки результата
Уникальность пользователей, сохранность выборки, min ≤ max, случаи с одной, несколькими и без подходящих оценок.

In [29]:
filled = sum(F.col(f"feat_min_watched_time_rating_{k}").isNotNull().cast("int") for k in (8, 9, 10))
checked = features.withColumn("ratings_with_data", filled)

print("строк:", features.count(), "| уникальных user_uid:", features.select("user_uid").distinct().count(),
      "| потеряно из выборки:", sample.join(features, "user_uid", "left_anti").count())
print("пар, где min > max:", features.filter(" OR ".join(
    f"feat_min_watched_time_rating_{k} > feat_max_watched_time_rating_{k}" for k in (8, 9, 10))).count())
print("пар, где заполнен только один из min/max:", features.filter(" OR ".join(
    f"(feat_min_watched_time_rating_{k} IS NULL) <> (feat_max_watched_time_rating_{k} IS NULL)" for k in (8, 9, 10))).count())
checked.groupBy("ratings_with_data").count().orderBy("ratings_with_data").show()

строк: 100 | уникальных user_uid: 100 | потеряно из выборки: 0
пар, где min > max: 0
пар, где заполнен только один из min/max: 0
+-----------------+-----+
|ratings_with_data|count|
+-----------------+-----+
|                0|   41|
|                1|   35|
|                2|   19|
|                3|    5|
+-----------------+-----+



Примеры: по одному пользователю на каждое число заполненных оценок. Для одного из них признаки пересчитаны вручную по сырым данным.

In [30]:
from pyspark.sql import Window
checked.withColumn("n", F.row_number().over(Window.partitionBy("ratings_with_data").orderBy("user_uid"))) \
       .filter("n = 1").drop("n").orderBy("ratings_with_data").show()

u = checked.filter("ratings_with_data = 3").orderBy("user_uid").first().user_uid
print(f"ручная проверка пользователя {u}: все его просмотры с оценками 8-10")
(ratings.filter((F.col("user_uid") == u) & F.col("rating").isin(8, 9, 10))
 .join(views, ["user_uid", "element_uid"]).groupBy("rating")
 .agg(F.min("watched_time"), F.max("watched_time"), F.count("*").alias("views")).orderBy("rating").show())

+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+-----------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|ratings_with_data|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+-----------------+
|    8504|                          NULL|                          NULL|                          NULL|                          NULL|                           NULL|                           NULL|                0|
|   55511|                          NULL|                          NULL|                          1293|                          129

Все 100 пользователей выборки на месте, повторов нет, min ≤ max везде и min/max заполняются только парой. У 41 пользователя нет подходящих просмотров - все 6 признаков NULL, у 35 есть данные по одной оценке, у 19 по двум, у 5 по всем трём. Ручная проверка пользователя 110138 по сырым данным совпала с его признаками.

In [31]:
spark.stop()